# Tipificación de casos de post-venta con LLM

Notebook de exploración. La lógica vive en `src/`; aquí solo se orquesta y se analiza.

**Flujo:** data dummy → preprocesamiento → LLM → evaluación → resultados.

In [1]:
import sys, json
sys.path.insert(0, '..') if 'notebooks' in sys.path[0] else sys.path.insert(0, '.')

import pandas as pd
from src.config import Settings, Taxonomy, DATA_RAW
from src.preprocessing import preprocesar
from src.llm.factory import get_llm_client
from src.classifier import Clasificador
from src.evaluation import evaluar_todo
from src.pipeline import ejecutar_pipeline

pd.set_option('display.max_colwidth', 80)

ModuleNotFoundError: No module named 'src'

## 1. Taxonomía (fuente única de verdad)
Cambiar la taxonomía = editar `config/taxonomy.yaml`. No se toca el código ni el prompt.

In [ ]:
tax = Taxonomy()
print(tax.version)
for c in tax.categorias:
    print(f'{c:20s} -> {tax.subcategorias(c)}')

## 2. Data dummy y preprocesamiento

In [ ]:
df = pd.read_csv(DATA_RAW / 'casos_postventa.csv')
print(df.shape)
df.head()

In [ ]:
df_prep, reporte = preprocesar(df)
print(json.dumps(reporte, indent=2))
df_prep[['id_caso','texto_caso','texto_llm','valido']].head()

## 3. Prompt y una clasificación individual
Cambia `provider` a `ollama` o `groq` cuando tengas el modelo listo; el resto del código no cambia.

In [ ]:
s = Settings(); s.provider = 'mock'   # mock | ollama | groq | openrouter
clf = Clasificador(get_llm_client(s), tax, s)
c = clf.clasificar_uno('DEMO_1', 'App', 'Hice una transferencia y me la cobraron dos veces')
c

## 4. Clasificación del lote completo

In [ ]:
r = ejecutar_pipeline(DATA_RAW / 'casos_postventa.csv', s=s)
res = r['resultados']
print(json.dumps(r['operativo'], indent=2, ensure_ascii=False))
res.head()

## 5. Evaluación

In [ ]:
ev = r['evaluacion']
print(json.dumps(ev['resumen'], indent=2, ensure_ascii=False))
display(ev['errores_top'])
display(ev['calibracion'])
display(ev['automatizacion'])

In [ ]:
import matplotlib.pyplot as plt
m = ev['matriz_categoria']
fig, ax = plt.subplots(figsize=(8,6))
im = ax.imshow(m.values, cmap='Blues')
ax.set_xticks(range(len(m.columns))); ax.set_xticklabels([c.split('::')[1] for c in m.columns], rotation=45, ha='right')
ax.set_yticks(range(len(m.index))); ax.set_yticklabels([i.split('::')[1] for i in m.index])
ax.set_title('Matriz de confusión - categoría'); fig.colorbar(im); plt.tight_layout()

## 6. Análisis de error: ¿dónde falla el modelo?
Revisa los casos mal clasificados con alta confianza: son los peligrosos.

In [ ]:
malos = res[(res['acierto_subcategoria'] == False) & (res['confianza'] >= 0.8)]
print(len(malos), 'errores con alta confianza')
malos[['texto_caso','subcategoria_real','subcategoria','confianza','justificacion']].head(15)

## 7. Siguientes pasos
1. Correr el mismo notebook con `provider='ollama'` y comparar contra el baseline `mock`.
2. Ajustar el prompt / few-shots con los errores del paso 6.
3. Calibrar el umbral con la curva de automatización según el costo de un error vs el costo de revisión humana.